In [0]:
%pip install newspaper3k
%pip install lxml_html_clean

In [0]:
import datetime
import time
import re
from pprint import pprint
from newspaper import Article, Config
import pandas as pd
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, date_format, col, expr, to_date

In [0]:
schema = StructType([
    StructField("date", StringType(), True),
    StructField("url", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("outletName", StringType(), True),
    StructField("outletLogo", StringType(), True),
    StructField("outletTwitter", StringType(), True),
    StructField("title", StringType(), True),
    StructField("image", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("lang", StringType(), True),
    StructField("author", StringType(), True),
])

In [0]:
unprocessed_files_df = spark.sql("""
    select file_name
    from news_app.default.gal_files
    where processed = false
    order by file_name
""").limit(1000)
# display(unprocessed_files_df)

file_paths = [f"/Volumes/news_app/default/news_app_volume/{row.file_name}" for row in unprocessed_files_df.collect()]

if file_paths:
    gal_df = spark.read.schema(schema).json(file_paths)
    # display(gal_df)
else:
    print("No unprocessed files found.")

In [0]:
outlet_names = [
    # "TV Guide", --Incomplete story
    "quicknews-africa.net",
    "The Nordic Page",
    "The Times of India",
    "Yahoo Finance",
    "DailyRidge.com - Fast-Factual-Free",
    # "BizToc", --Incomplete story
    "The Indian Express",
    "The Star",
    "The Economic Times",
    "Hindustan Times",
    "Mail Online",
    "Express.co.uk",
    "Free Press Journal",
    "Daily Mirror",
    "Moneycontrol",
    "The Hindu",
    "Yahoo News",
    "mint",
    "ABC News",
    "the Guardian",
    "Zee News",
    "The Mail",
    "bbc.com",
    "Fox News",
    "New York Post",
    "Middle East Star"
]

gal_df = gal_df.filter(gal_df['lang'] == 'en')
gal_df = gal_df.filter(gal_df.outletName.isin(outlet_names))

In [0]:
def parse_article(row):
    url = row['url']
    result = {
        'news_text': None,
        'image_url': None,
        'keywords': [],
        'published_timestamp': None
    }

    config = Config()
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
    
    article = Article(url)
    article.download()
    article.parse()
    # article.nlp()
    result['news_text'] = article.text
    result['image_url'] = article.top_image
    result['keywords'] = article.keywords if isinstance(article.keywords, list) else []
    result['published_timestamp'] = article.publish_date

    return result

pdf = gal_df.toPandas()

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

row_count = 0
success_count = 0
failure_count = 0
news_list = []

def process_row(row):
    row_dict = row.to_dict()
    try:
        result = {**row_dict, **parse_article(row_dict)}
        return (row_dict['url'], result, True)
    except Exception:
        return (row_dict['url'], None, False)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(process_row, row): idx for idx, row in pdf.iterrows()}
    for future in as_completed(futures):
        url, result, success = future.result()
        row_count += 1
        print(row_count, url)
        if success:
            news_list.append(result)
            success_count += 1
        else:
            failure_count += 1

print(row_count, success_count, failure_count)

In [0]:
schema = gal_df.schema

# print(schema)
news_schema = [
    StructField('news_text', StringType()),
    StructField('image_url', StringType()),
    StructField('keywords', ArrayType(StringType())),
    StructField('published_timestamp', TimestampType())
]

for field in news_schema:
    schema = schema.add(field)

# print(schema)
gal_processed_df = spark.createDataFrame(news_list, schema=schema)
gal_processed_df = gal_processed_df.where("news_text is not null and length(desc) < length(news_text)")
gal_processed_df = gal_processed_df.withColumn(
    "added_timestamp",
    date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'")
)

gal_processed_df = gal_processed_df.withColumn("partition_date", to_date("added_timestamp"))
gal_processed_df = gal_processed_df.withColumn("article_id", expr("uuid()"))
cols = ["article_id"] + [col for col in gal_processed_df.columns if col != "article_id"]
gal_processed_df = gal_processed_df.select(*cols)

display(gal_processed_df)
gal_processed_df.count()

In [0]:
gal_processed_df.write.mode("append").option("mergeSchema", "true").partitionBy("partition_date").saveAsTable("news_app.default.gdelt_gal")